In [ ]:
!pip install transformers datasets torch accelerate


  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
Using cached datasets-4.5.0-py3-none-any.whl (515 kB)
Using cached multiprocess-0.70.18-py313-none-any.whl (151 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl (31 kB)

   ---------- ----------------------------- 1/4 [multiprocess]
   -------------------- ------------------- 2/4 [accelerate]
   -------------------- ------------------- 2/4 [accelerate]
   ------------------------------ --------- 3/4 [datasets]
   ------------------------------ --------- 3/4 [datasets]
   ------------------------------ --------- 3/4 [datasets]
   ------------------------------ --------- 3/4 [datasets]
   ---------------------------------------- 4/4 [datasets]



In [8]:
!pip install "transformers[torch]"

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)


In [4]:
# from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))


Embedding(50257, 768)

In [5]:
# from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "stories.txt"})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenization complete!")
print(tokenized_dataset)


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenization complete!
DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 7
    })
})


In [6]:
# Data Collator (Batching for GPT-2)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(" Data collator created!")


 Data collator created!


In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-stories",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=torch.cuda.is_available()
)

print(" Training arguments set!")
print(TrainingArguments.fp16)


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

print("Trainer initialized!")

Trainer initialized!


In [ ]:
trainer.train()

d:\TY VIIT\SEM VI\GenAI-Assignments\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.41s/it]


TrainOutput(global_step=6, training_loss=3.6417083740234375, metrics={'train_runtime': 99.7467, 'train_samples_per_second': 0.211, 'train_steps_per_second': 0.06, 'total_flos': 5487132672000.0, 'train_loss': 3.6417083740234375, 'epoch': 3.0})

In [ ]:
# Save Fine-Tuned Model
model.save_pretrained("./gpt2-finetuned-stories")
tokenizer.save_pretrained("./gpt2-finetuned-stories")

print("Fine-tuning complete! Model saved.")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]

Fine-tuning complete! Model saved.


Story Generation After Training

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import textwrap

model_name = "./gpt2-finetuned-stories"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Fine-tuned model loaded for story generation!")


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
# Story Generator Function
def generate_story(prompt, max_length=250):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    story = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Formatting (no right-side scrolling)
    story = story.replace(". ", ".\n")
    return textwrap.fill(story, width=90)


In [ ]:
# Generate Story Output
story_prompt = "The spaceship landed silently on Mars,"
print(generate_story(story_prompt))


The spaceship landed silently on Mars, in a region of the solar system where it is
currently orbiting. The researchers are now trying to find out what caused its massive
surface features to fall and how they have changed over time," said lead author Dr Steve
Schmitt, from Columbia University's School for Earth-based Systems Analysis and
Exploration (SESA) who led the study published this week online October 12 in PLOS One .
 "We believe that some or all parts may be involved because there was an ancient presence
here but we think these were not yet fully formed before our arrival." One major
hypothesis has been made by geologists looking into possible causes - if so why? While
many scientists question whether life could exist outside outer space beyond Neptune ,
their main conclusion being simple: "It looks like most things don't go as planned". Even
though such hypotheses assume no intelligent planets would emerge; NASA astronomers say
otherwise... [Read more about] This piece origina